# Scenario Analysis

AbaQuant treats scenario analysis as a first-class workflow across three
domains, all following the same pattern:

```
base case -> scenario grid -> visual report
```

**Sections:**
1. Derivative spot–volatility scenario grid
2. Portfolio one-period asset-shock scenario
3. Credit debt/EBITDA multiplier scenario
4. Summary and figures


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import pandas as pd

from abaquant.credit import (
    BalanceSheetInputs,
    CashFlowInputs,
    CreditAnalysisInputs,
    CreditHistoricalSeries,
    IncomeStatementInputs,
    MarketEquityObservation,
    PriorPeriodInputs,
    calculate_credit_proxy_metrics,
)
from abaquant.derivatives.models import BlackScholesMertonModel
from abaquant.portfolio import PortfolioAllocator
from abaquant.visualization import VisualizationError

## 1. Derivative spot–volatility scenario grid

Evaluate a call option's price and Greeks across a grid of spot prices and
volatilities — useful for stress-testing a position ahead of an earnings
announcement or macro event.


In [ ]:
option_model = BlackScholesMertonModel(
    spot_price=100.0,
    strike_price=105.0,
    maturity_years=1.0,
    risk_free_rate=0.05,
    volatility=0.22,
)
derivative_grid = option_model.scenario_grid(
    spot_prices=[80.0, 90.0, 100.0, 110.0, 120.0],
    volatilities=[0.15, 0.20, 0.25, 0.30],
    option_type="call",
)
derivative_grid.data.head()

## 2. Portfolio one-period asset-shock scenario

Apply an explicit percentage shock to each asset in a maximum-Sharpe
allocation and see the resulting portfolio return and ending value.


In [ ]:
returns = pd.DataFrame(
    {
        "ALPHA": [0.01, -0.002, 0.006, 0.004, 0.003],
        "BETA": [0.003, 0.005, -0.001, 0.002, 0.004],
        "GAMMA": [-0.002, 0.007, 0.004, 0.006, 0.001],
    }
)
allocator = PortfolioAllocator(returns, annual_risk_free_rate=0.02)
weights = allocator.mean_variance.maximum_sharpe()

portfolio_scenario = allocator.scenario_analysis(
    shocks={"ALPHA": -0.20, "BETA": -0.10, "GAMMA": -0.15},
    weights=weights,
    base_value=1_000_000.0,
)
print(f"Portfolio scenario return: {portfolio_scenario.portfolio_return:.4%}")
print(f"Ending value:              {portfolio_scenario.ending_value:,.2f}")

## 3. Credit debt/EBITDA multiplier scenario

Stress a fundamentals-based credit-proxy assessment by scaling debt and
EBITDA independently across a small grid, to see which input dominates the
synthetic score.


In [ ]:
assessment = calculate_credit_proxy_metrics(
    CreditAnalysisInputs(
        balance_sheet=BalanceSheetInputs(
            total_debt=120.0, total_equity=300.0, current_assets=180.0, inventory=20.0,
            current_liabilities=90.0, cash_and_cash_equivalents=35.0, total_assets=520.0,
            total_liabilities=220.0, retained_earnings=125.0, long_term_debt=105.0,
            shares_outstanding=100.0,
        ),
        income_statement=IncomeStatementInputs(
            revenue=700.0, gross_profit=310.0, ebit=90.0, ebitda=115.0,
            interest_expense=9.0, net_income=62.0,
        ),
        cash_flow_statement=CashFlowInputs(operating_cash_flow=78.0),
        prior_period=PriorPeriodInputs(
            total_assets=500.0, net_income=55.0, long_term_debt=112.0, current_assets=170.0,
            current_liabilities=95.0, shares_outstanding=100.0, gross_profit=290.0, revenue=660.0,
        ),
        market_equity=MarketEquityObservation(market_value_equity=950.0),
        historical_series=CreditHistoricalSeries(
            earnings_history=(46.0, 51.0, 55.0, 62.0),
            leverage_history=(0.54, 0.48, 0.43, 0.40),
        ),
    )
)
credit_grid = assessment.scenario_analysis(
    debt_multiplier=[1.0, 1.25, 1.50],
    ebitda_multiplier=[1.0, 0.75, 0.50],
)
credit_grid.data

## 4. Summary and figures

Compact scalar highlights from all three scenario families, plus their visual reports.


In [ ]:
summary = {
    "derivative_highest_call_price": float(derivative_grid.data["price"].max()),
    "derivative_lowest_delta": float(derivative_grid.data["delta"].min()),
    "portfolio_scenario_return": portfolio_scenario.portfolio_return,
    "portfolio_ending_value": portfolio_scenario.ending_value,
    "credit_lowest_proxy_score": float(credit_grid.data["synthetic_credit_proxy_score"].min()),
}
for key, value in summary.items():
    print(f"{key:34s}: {value}")

In [ ]:
try:
    figures = {
        "derivative_price_surface": derivative_grid.visualize(metric="price", chart="surface"),
        "derivative_delta_heatmap": derivative_grid.visualize(metric="delta", chart="heatmap"),
        "portfolio_contributions": portfolio_scenario.visualize(chart="contributions"),
        "portfolio_shocks": portfolio_scenario.visualize(chart="shocks"),
        "credit_score_heatmap": credit_grid.visualize(
            metric="synthetic_credit_proxy_score", chart="heatmap"
        ),
        "credit_net_debt_curves": credit_grid.visualize(
            metric="net_debt_to_ebitda", chart="curves"
        ),
    }
    print(f"Created {len(figures)} scenario figures: {list(figures)}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Scenario grids give you a structured way to ask "what if" across
derivatives, portfolios, and credit — without hand-rolling nested loops.
Every result object exposes `.data` (a tidy DataFrame) plus `.visualize()`
for a quick heatmap, surface, or curve chart.
